In [1]:
# Align company and macro observations to each prediction date.
# Apply the financial reporting lag, minimum market history, and no-future-data rules.

In [2]:
# Locate shared project configuration and reusable helpers.
import sys as _sys
from pathlib import Path as _Path

_UTILS_ROOT = next(
    path
    for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
    if (path / "notebook_utils.py").is_file()
)
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/clean_company_long.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
MATCH_RATIO = 2
REPORTING_LAG_DAYS = 90
MIN_MARKET_PRICE_MONTHS = 13
MAX_MARKET_AGE_DAYS = 45

matches = pd.read_csv(
    PROCESSED / f"matched_event_sample_1to{MATCH_RATIO}.csv",
    parse_dates=["bankrupt_prediction_date", "control_pseudo_event_date"],
)
metadata = pd.read_csv(PROCESSED / "company_availability.csv")
bankrupt_events = pd.read_csv(
    PROCESSED / "bankruptcy_events.csv",
    parse_dates=["prediction_date", "outcome_end_date"],
)
control_events = pd.read_csv(
    PROCESSED / "nonbankruptcy_candidate_events.csv",
    parse_dates=["prediction_date", "outcome_end_date"],
)
if matches.empty:
    raise ValueError("The matched sample is empty; inspect notebook 06")
if metadata["company_id"].duplicated().any():
    raise ValueError("Duplicate company metadata")
positive_keys = matches[["bankrupt_company_id", "bankrupt_prediction_date"]].rename(
    columns={
        "bankrupt_company_id": "company_id",
        "bankrupt_prediction_date": "prediction_date",
    }
)
negative_keys = matches[["control_company_id", "control_pseudo_event_date"]].rename(
    columns={
        "control_company_id": "company_id",
        "control_pseudo_event_date": "prediction_date",
    }
)
keys = pd.concat([positive_keys, negative_keys], ignore_index=True)
link_counts = (
    keys.groupby(["company_id", "prediction_date"])
    .size()
    .rename("matched_pair_links")
    .reset_index()
)
event_registry = pd.concat([bankrupt_events, control_events], ignore_index=True)
if event_registry.duplicated(["company_id", "prediction_date"]).any():
    raise ValueError("Duplicate company-events in notebook 05 outputs")
events = link_counts.merge(
    event_registry,
    on=["company_id", "prediction_date"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
if not events["_merge"].eq("both").all():
    raise ValueError("Matched events absent from notebook 05 outputs")
events = events.drop(columns="_merge").merge(
    metadata[
        [
            "company_id",
            "company_type",
            "ticker",
            "name",
            "bloomberg_identifier",
            "bloomberg_sector",
            "bloomberg_industry",
        ]
    ],
    on="company_id",
    how="left",
    validate="many_to_one",
)
if events[["company_id", "prediction_date", "bloomberg_identifier"]].isna().any().any():
    raise ValueError("Missing event identifiers, dates, or metadata")
if not events["prediction_year"].eq(events["prediction_date"].dt.year).all():
    raise ValueError("Prediction year/date mismatch")
events = events.sort_values(
    ["prediction_date", "company_id"], kind="stable"
).reset_index(drop=True)
events["event_id"] = (
    events["company_id"] + "@" + events["prediction_date"].dt.strftime("%Y-%m-%d")
)
events["financial_cutoff_date"] = events["prediction_date"] - pd.Timedelta(
    days=REPORTING_LAG_DAYS
)

In [3]:
# Load histories for matched companies only, preserving all field dates and NaNs.
company_ids = set(events["company_id"])
company_chunks = []
for chunk in pd.read_csv(
    PROCESSED / "clean_company_long.csv",
    usecols=["company_id", "date", "field", "data_type", "value"],
    chunksize=250_000,
):
    company_chunks.append(chunk.loc[chunk["company_id"].isin(company_ids)].copy())
company_long = pd.concat(company_chunks, ignore_index=True)
company_long["date"] = pd.to_datetime(
    company_long["date"], errors="raise"
).dt.normalize()
company_long["value"] = pd.to_numeric(company_long["value"], errors="raise")
if company_long["date"].isna().any():
    raise ValueError("Missing company observation dates")
if company_long.duplicated(["company_id", "date", "field"]).any():
    raise ValueError("Duplicate company/date/field records; rerun cleaning")
if not company_long["data_type"].isin(["financial", "market"]).all():
    raise ValueError("Unknown company field type")
if company_long.groupby("field")["data_type"].nunique().gt(1).any():
    raise ValueError("A field has multiple data types")
if not np.isfinite(company_long.loc[company_long["value"].notna(), "value"]).all():
    raise ValueError("Nonfinite company values; rerun cleaning")
financial_fields = sorted(
    company_long.loc[company_long["data_type"].eq("financial"), "field"].unique()
)
market_fields = sorted(
    company_long.loc[company_long["data_type"].eq("market"), "field"].unique()
)
if "total_assets" not in financial_fields or "adjusted_price" not in market_fields:
    raise ValueError("Assets and adjusted prices are required")
financial_long = company_long.loc[company_long["data_type"].eq("financial")].copy()
market_long = company_long.loc[company_long["data_type"].eq("market")].copy()

macro_long = pd.read_csv(
    PROCESSED / "clean_macro_long.csv", usecols=["date", "field", "value"]
)
macro_long["date"] = pd.to_datetime(macro_long["date"], errors="raise").dt.normalize()
macro_long["value"] = pd.to_numeric(macro_long["value"], errors="raise")
if macro_long["date"].isna().any():
    raise ValueError("Missing macro observation dates")
if macro_long.duplicated(["date", "field"]).any():
    raise ValueError("Duplicate macro date/field records; rerun cleaning")
if not np.isfinite(macro_long.loc[macro_long["value"].notna(), "value"]).all():
    raise ValueError("Nonfinite macro values; rerun cleaning")
macro_fields = sorted(macro_long["field"].unique())

In [4]:
# Select a coherent statement row, preserving unavailable fields at that date.
def select_financial_statements(event_rows, history, fields, lag_days):
    result = (
        event_rows[["event_id", "company_id", "prediction_date"]]
        .copy()
        .reset_index(drop=True)
    )
    result["financial_cutoff_date"] = result["prediction_date"] - pd.Timedelta(
        days=lag_days
    )
    reference_dates = np.full(
        len(result), np.datetime64("NaT", "ns"), dtype="datetime64[ns]"
    )
    anchors = history.loc[history["value"].notna()].drop_duplicates(
        ["company_id", "date"]
    )
    anchor_dates = {
        company: group["date"].sort_values().to_numpy(dtype="datetime64[ns]")
        for company, group in anchors.groupby("company_id", sort=False)
    }
    for company, indices in result.groupby("company_id", sort=False).indices.items():
        observed = anchor_dates.get(company, np.array([], dtype="datetime64[ns]"))
        cutoffs = result.loc[indices, "financial_cutoff_date"].to_numpy(
            dtype="datetime64[ns]"
        )
        positions = np.searchsorted(observed, cutoffs, side="right") - 1
        usable = positions >= 0
        if usable.any():
            reference_dates[indices[usable]] = observed[positions[usable]]
    result["financial_reference_date"] = reference_dates
    wide = (
        history.pivot(index=["company_id", "date"], columns="field", values="value")
        .reindex(columns=fields)
        .reset_index()
    )
    wide.columns.name = None
    wide = wide.rename(columns={"date": "financial_reference_date"})
    result = result.merge(
        wide,
        on=["company_id", "financial_reference_date"],
        how="left",
        validate="many_to_one",
    )
    result["financial_field_count"] = result[fields].notna().sum(axis=1)
    result["financial_age_days"] = (
        result["prediction_date"] - result["financial_reference_date"]
    ).dt.days.astype("Int64")
    result["days_between_prediction_and_financial_statement"] = result[
        "financial_age_days"
    ]
    return result


financial_snapshots = select_financial_statements(
    events, financial_long, financial_fields, REPORTING_LAG_DAYS
)

In [5]:
# Per-field as-of selection keeps differing market/macro dates explicit.
def select_asof_fields(event_rows, history, fields, kind, max_age_days=None):
    result = (
        event_rows[["event_id", "company_id", "prediction_date"]]
        .copy()
        .reset_index(drop=True)
    )
    usable = history.loc[history["value"].notna()].copy()
    grouped = {}
    keys = ["company_id", "field"] if kind == "market" else ["field"]
    for key, group in usable.groupby(keys, sort=False):
        if kind == "macro" and isinstance(key, tuple):
            key = key[0]
        group = group.sort_values("date")
        grouped[key] = (
            group["date"].to_numpy(dtype="datetime64[ns]"),
            group["value"].to_numpy(dtype=float),
        )
    reference_columns = []
    for field in fields:
        values = np.full(len(result), np.nan)
        dates = np.full(len(result), np.datetime64("NaT", "ns"), dtype="datetime64[ns]")
        event_groups = (
            result.groupby("company_id", sort=False).indices
            if kind == "market"
            else {None: np.arange(len(result))}
        )
        for company_id, indices in event_groups.items():
            observed, observations = grouped.get(
                (company_id, field) if kind == "market" else field,
                (np.array([], dtype="datetime64[ns]"), np.array([], dtype=float)),
            )
            query = result.loc[indices, "prediction_date"].to_numpy(
                dtype="datetime64[ns]"
            )
            positions = np.searchsorted(observed, query, side="right") - 1
            available = positions >= 0
            if available.any():
                candidate_dates = observed[positions[available]]
                chosen = indices[available]
                if max_age_days is not None:
                    fresh = candidate_dates >= query[available] - np.timedelta64(
                        max_age_days, "D"
                    )
                    chosen = chosen[fresh]
                    candidate_dates = candidate_dates[fresh]
                    candidate_values = observations[positions[available]][fresh]
                else:
                    candidate_values = observations[positions[available]]
                values[chosen] = candidate_values
                dates[chosen] = candidate_dates
        result[field] = values
        reference_column = f"{kind}_{field}_reference_date"
        result[reference_column] = dates
        result[f"{kind}_{field}_days_before_prediction"] = (
            result["prediction_date"] - result[reference_column]
        ).dt.days.astype("Int64")
        reference_columns.append(reference_column)
    result[f"{kind}_reference_date"] = result[reference_columns].max(axis=1)
    result[f"{kind}_age_days"] = (
        result["prediction_date"] - result[f"{kind}_reference_date"]
    ).dt.days.astype("Int64")
    result[f"{kind}_field_count"] = result[fields].notna().sum(axis=1)
    if kind == "market":
        result["days_between_prediction_and_market_data"] = result["market_age_days"]
    return result


market_snapshots = select_asof_fields(
    events, market_long, market_fields, "market", MAX_MARKET_AGE_DAYS
)
macro_snapshots = select_asof_fields(events, macro_long, macro_fields, "macro")

In [6]:
# Preserve enough dated monthly prices to compute later return features.
def select_monthly_price_history(event_rows, history, minimum_months, max_age_days):
    result = (
        event_rows[["event_id", "company_id", "prediction_date"]]
        .copy()
        .reset_index(drop=True)
    )
    valid_prices = history.loc[
        history["field"].eq("adjusted_price") & history["value"].gt(0)
    ]
    prices = {}
    for company_id, group in valid_prices.groupby("company_id", sort=False):
        group = group.sort_values("date")
        prices[company_id] = (
            group["date"].to_numpy(dtype="datetime64[ns]"),
            group["value"].to_numpy(dtype=float),
        )
    lag_values = np.full((len(result), minimum_months), np.nan)
    lag_dates = np.full(
        (len(result), minimum_months),
        np.datetime64("NaT", "ns"),
        dtype="datetime64[ns]",
    )
    month_counts = np.zeros(len(result), dtype=int)
    used_counts = np.zeros(len(result), dtype=int)
    contiguous = np.zeros(len(result), dtype=bool)
    recent = np.zeros(len(result), dtype=bool)
    for index, event in enumerate(result.itertuples(index=False)):
        observed, values = prices.get(
            event.company_id,
            (np.array([], dtype="datetime64[ns]"), np.array([], dtype=float)),
        )
        prediction = event.prediction_date.to_datetime64()
        stop = np.searchsorted(observed, prediction, side="right")
        if not stop:
            continue
        before = observed[:stop]
        months = before.astype("datetime64[M]").astype(np.int64)
        # Last available price in each calendar month, restricted to prediction.
        monthly_positions = np.flatnonzero(np.r_[months[:-1] != months[1:], True])
        month_counts[index] = len(monthly_positions)
        window_positions = monthly_positions[-minimum_months:]
        used_counts[index] = len(window_positions)
        window_months = months[window_positions]
        contiguous[index] = len(window_positions) == minimum_months and np.diff(
            window_months
        ).astype(int).tolist() == [1] * (minimum_months - 1)
        recent[index] = before[-1] >= prediction - np.timedelta64(max_age_days, "D")
        latest_first = window_positions[::-1]
        lag_values[index, : len(latest_first)] = values[latest_first]
        lag_dates[index, : len(latest_first)] = observed[latest_first]
    for lag in range(minimum_months):
        result[f"market_monthly_price_{lag:02d}"] = lag_values[:, lag]
        result[f"market_monthly_price_date_{lag:02d}"] = lag_dates[:, lag]
    result["market_history_months_before"] = month_counts
    result["market_history_months_used"] = used_counts
    result["market_history_contiguous"] = contiguous
    result["market_history_recent"] = recent
    result["market_history_eligible"] = contiguous & recent
    result["market_history_start_date"] = pd.DataFrame(lag_dates).min(axis=1)
    result["market_history_end_date"] = pd.DataFrame(lag_dates).max(axis=1)
    return result


market_history = select_monthly_price_history(
    events, market_long, MIN_MARKET_PRICE_MONTHS, MAX_MARKET_AGE_DAYS
)

In [7]:
# Consolidate one row per unique company-event, without duplicating reused controls.
company_alignment = (
    events.merge(
        financial_snapshots.drop(
            columns=["company_id", "prediction_date", "financial_cutoff_date"]
        ),
        on="event_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        market_snapshots.drop(columns=["company_id", "prediction_date"]),
        on="event_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        market_history.drop(columns=["company_id", "prediction_date"]),
        on="event_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        macro_snapshots[
            ["event_id", "macro_reference_date", "macro_age_days", "macro_field_count"]
        ],
        on="event_id",
        how="left",
        validate="one_to_one",
    )
)
company_alignment["has_financial_statement"] = company_alignment[
    "financial_reference_date"
].notna()
company_alignment["has_market_snapshot"] = company_alignment[
    "market_reference_date"
].notna()
company_alignment["has_macro_observation"] = company_alignment[
    "macro_reference_date"
].notna()
company_alignment["alignment_eligible"] = company_alignment[
    [
        "has_financial_statement",
        "has_market_snapshot",
        "has_macro_observation",
        "market_history_eligible",
    ]
].all(axis=1)
alignment_summary = company_alignment.groupby("target").agg(
    input_events=("event_id", "size"),
    retained_events=("alignment_eligible", "sum"),
    lagged_statements=("has_financial_statement", "sum"),
    market_snapshots=("has_market_snapshot", "sum"),
    sufficient_market_history=("market_history_eligible", "sum"),
    macro_observations=("has_macro_observation", "sum"),
)

company_aligned = company_alignment.loc[company_alignment["alignment_eligible"]].copy()
company_aligned = company_aligned.drop(
    columns=[
        "has_financial_statement",
        "has_market_snapshot",
        "has_macro_observation",
        "alignment_eligible",
    ]
)
company_aligned = company_aligned.sort_values(
    ["prediction_date", "company_id"], kind="stable"
).reset_index(drop=True)
AUDIT_COLUMNS = [
    "financial_cutoff_date",
    "financial_reference_date",
    "market_reference_date",
    "macro_reference_date",
    "prediction_date",
    "financial_age_days",
    "market_age_days",
    "macro_age_days",
    "days_between_prediction_and_financial_statement",
    "days_between_prediction_and_market_data",
]
EVENT_COLUMNS = [
    "event_id",
    "company_id",
    "target",
    "event_type",
    "prediction_year",
    "outcome_end_date",
]
macro_aligned = company_aligned[EVENT_COLUMNS + AUDIT_COLUMNS].merge(
    macro_snapshots.drop(
        columns=[
            "company_id",
            "prediction_date",
            "macro_reference_date",
            "macro_age_days",
        ]
    ),
    on="event_id",
    how="left",
    validate="one_to_one",
)
retained_keys = set(
    zip(company_aligned["company_id"], company_aligned["prediction_date"])
)
matched_set_audit = matches.copy()
matched_set_audit["bankrupt_event_retained"] = [
    (row.bankrupt_company_id, row.bankrupt_prediction_date) in retained_keys
    for row in matches.itertuples(index=False)
]
matched_set_audit["control_event_retained"] = [
    (row.control_company_id, row.control_pseudo_event_date) in retained_keys
    for row in matches.itertuples(index=False)
]
matched_set_audit["pair_retained"] = matched_set_audit[
    ["bankrupt_event_retained", "control_event_retained"]
].all(axis=1)
set_coverage = matched_set_audit.groupby(
    ["bankrupt_company_id", "bankrupt_prediction_date"]
)["pair_retained"].sum()

In [8]:
# Enforce the reporting lag and as-of rules for every retained event.
assert (
    company_aligned["financial_reference_date"]
    <= company_aligned["financial_cutoff_date"]
).all()
assert (
    company_aligned["financial_cutoff_date"]
    == company_aligned["prediction_date"] - pd.Timedelta(days=REPORTING_LAG_DAYS)
).all()
assert (
    company_aligned["financial_reference_date"] <= company_aligned["prediction_date"]
).all()
assert (
    company_aligned["market_reference_date"] <= company_aligned["prediction_date"]
).all()
assert (
    company_aligned["macro_reference_date"] <= company_aligned["prediction_date"]
).all()
assert (macro_aligned["macro_reference_date"] <= macro_aligned["prediction_date"]).all()

# Check every populated per-field reference and monthly price date as well.
for frame, prefix, fields in [
    (company_aligned, "market", market_fields),
    (macro_aligned, "macro", macro_fields),
]:
    for field in fields:
        dates = frame[f"{prefix}_{field}_reference_date"]
        used = frame[field].notna()
        assert dates.loc[used].le(frame.loc[used, "prediction_date"]).all()

price_date_columns = [
    f"market_monthly_price_date_{lag:02d}" for lag in range(MIN_MARKET_PRICE_MONTHS)
]
for column in price_date_columns:
    assert company_aligned[column].le(company_aligned["prediction_date"]).all()

# Keep the legacy day-gap columns aligned with the new age fields.
assert company_aligned["financial_age_days"].equals(
    company_aligned["days_between_prediction_and_financial_statement"]
)
assert company_aligned["market_age_days"].equals(
    company_aligned["days_between_prediction_and_market_data"]
)

# Save the aligned company and macro tables for the feature notebooks.
company_output_path = PROCESSED / "event_aligned_company_data.csv"
macro_output_path = PROCESSED / "event_aligned_macro_data.csv"
company_aligned.to_csv(company_output_path, index=False, date_format="%Y-%m-%d")
macro_aligned.to_csv(macro_output_path, index=False, date_format="%Y-%m-%d")